# Week 2: Factor Foundations — Value and Momentum

Research question: *From July 2021 to August 2026, among my 50 large-cap stocks, which earns the higher average monthly return: high or low book-to-market stocks (sorted annually each June), and past winners or past losers (12-1 momentum, sorted monthly), using equal-weight quintile portfolios?*

Design note and look-ahead checklist: see Task 1 (Google Doc).

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
import numpy as np
import pandas as pd
from datetime import date

sys.path.append("../../src")
from data_utils import (load_ohlcv, check_missing_prices, check_duplicates, check_zero_prices,
                        check_extreme_returns, check_invalid_dates,
                        load_book_equity, get_shares_history, get_splits_history)
from factor_utils import get_ranking_dates, value_signal, detect_shares_split_adjusted

RAW = "../../data/raw"
SIGNALS = "../../data/signals"
os.makedirs(RAW, exist_ok=True)
os.makedirs(SIGNALS, exist_ok=True)

tickers = [
    "NVDA", "META", "GOOGL", "MSFT", "TSLA", "ORCL", "AMD", "INTC", "SMCI", "MU",
    "RKLB", "WDC", "STX", "CRWD", "PLTR", "AAPL", "AMZN", "AVGO", "CSCO", "ADBE",
    "CRM", "NFLX", "QCOM", "TXN", "TSM", "JPM", "BAC", "WFC", "V", "MA",
    "JNJ", "PFE", "MRK", "UNH", "ABT", "PG", "KO", "PEP", "WMT", "COST",
    "HD", "NKE", "MCD", "ASML", "XOM", "CVX", "HON", "CAT", "GE", "TMO",
]

## Task 2: Value Signal Construction

### 2.1 Prices from 2020
The June 2021 sort needs December 2020 prices. Downloaded once, then loaded from `data/raw/`.

In [ ]:
path = f"{RAW}/prices_2020.csv"
if os.path.exists(path):
    prices = pd.read_csv(path, parse_dates=["date"])
else:
    prices = load_ohlcv(tickers, start="2020-01-01", end=str(date.today()))
    prices["date"] = pd.to_datetime(prices["date"])
    if prices["date"].dt.tz is not None:
        prices["date"] = prices["date"].dt.tz_localize(None)
    prices.to_csv(path, index=False)

prices.groupby("ticker")["date"].min().sort_values().tail()   # latest first dates (PLTR, RKLB)

### 2.2 Data quality checks (reused from Week 1)

In [ ]:
print("Missing prices:", len(check_missing_prices(prices)))
print("Duplicates:", len(check_duplicates(prices)))
print("Zero prices:", len(check_zero_prices(prices)))
print("Invalid dates:", len(check_invalid_dates(prices, "2020-01-01", str(date.today()))))
check_extreme_returns(prices)[["date", "ticker", "daily_return"]]

### 2.3 Book equity from SEC EDGAR
As originally reported in each 10-K, with the filing date. **Fill in `SEC_HEADERS` in `data_utils.py` with your name and email first.**

In [ ]:
path = f"{RAW}/book_equity.csv"
if os.path.exists(path):
    book_equity = pd.read_csv(path, parse_dates=["fy_end", "be_filed"])
else:
    book_equity = load_book_equity(tickers)
    book_equity.to_csv(path, index=False)

# Force date types (an empty SEC result can turn these into plain objects)
book_equity["fy_end"] = pd.to_datetime(book_equity["fy_end"])
book_equity["be_filed"] = pd.to_datetime(book_equity["be_filed"])

be_hist = {tk: g.drop(columns="ticker").reset_index(drop=True) for tk, g in book_equity.groupby("ticker")}
print("Tickers with book equity:", len(be_hist))
print("No book equity returned:", sorted(set(tickers) - set(be_hist) - {"TSM", "ASML"}))

# Latest fiscal year available per ticker - anything stuck before 2025 is a data gap
latest = book_equity.groupby("ticker")["fy_end"].max().sort_values()
print("Latest FY before 2025 (data gaps):", latest[latest < "2025-01-01"].dt.date.to_dict())
book_equity[book_equity["ticker"] == "MCD"].tail(6)     # should be negative

**Diagnose book equity gaps.** For any ticker still missing, list every equity label it has used in 10-Ks since 2019. If a sensible label (total equity of the company's shareholders) shows up that isn't in `EQUITY_CONCEPTS`, add it to that list in `data_utils.py`, delete `data/raw/book_equity.csv`, and rerun 2.3.

In [ ]:
from data_utils import get_cik_map, find_equity_concepts
cik_map = get_cik_map()
for tk in ["ABT", "KO", "XOM", "V"]:
    print(f"\n===== {tk} =====")
    display(find_equity_concepts(cik_map[tk]).head(8))

### 2.4 Shares outstanding and split history (yfinance)

In [ ]:
sh_path, sp_path = f"{RAW}/shares.csv", f"{RAW}/splits.csv"
if os.path.exists(sh_path) and os.path.exists(sp_path):
    sh = pd.read_csv(sh_path, parse_dates=["date"])
    shares_hist = {tk: g.set_index("date")["shares"] for tk, g in sh.groupby("ticker")}
    sp = pd.read_csv(sp_path, parse_dates=["date"])
    sp["date"] = sp["date"].dt.normalize()          # yfinance stamps splits at 09:30; keep the date
    splits_hist = {tk: g.set_index("date")["ratio"] for tk, g in sp.groupby("ticker")}
else:
    shares_hist = get_shares_history(tickers, start="2020-06-01")
    splits_hist = get_splits_history(tickers)
    pd.concat(shares_hist, names=["ticker", "date"]).rename("shares").reset_index().to_csv(sh_path, index=False)
    pd.concat({k: v for k, v in splits_hist.items() if v is not None and len(v)},
              names=["ticker", "date"]).rename("ratio").reset_index().to_csv(sp_path, index=False)

print("Tickers with shares data:", len(shares_hist))

**Are yfinance share counts split-adjusted?** Detected automatically: for each real stock split (e.g. NVDA 10:1 in 2024, GOOGL/AMZN 20:1 in 2022), compare the median share count over the year before vs the year after. A jump by the split ratio means yfinance reports actual historical counts, so the split-adjusted `close` is converted back to the actual traded price to match. (Small yfinance "splits" like GE 1.28 or PFE 1.05 are spin-offs and are skipped.)

In [ ]:
SHARES_SPLIT_ADJUSTED, split_evidence = detect_shares_split_adjusted(shares_hist, splits_hist)
if SHARES_SPLIT_ADJUSTED is None:
    raise ValueError("No split found inside the share history - can't tell how shares are reported. Check manually.")
print("Share counts split-adjusted?", SHARES_SPLIT_ADJUSTED)
split_evidence

### 2.5 Ranking dates: last trading day of each June, 2021–2026

In [ ]:
ranking_dates = get_ranking_dates(prices, month=6, start="2021-01-01")
[d.date() for d in ranking_dates]

### 2.6 Test one date against the Task 1 worked example (t = 2023-06-30)
Expected: `fy_end` JPM 2022-12-31, AAPL 2022-09-24, MSFT 2022-06-30, NVDA 2022-01-30; `me_date` 2022-12-30; every `be_filed` before 2023-06-30; JPM flagged "bank".

In [ ]:
v23 = value_signal(pd.Timestamp("2023-06-30"), tickers, be_hist, shares_hist, splits_hist, prices,
                   shares_split_adjusted=SHARES_SPLIT_ADJUSTED)
v23.set_index("ticker").loc[["JPM", "AAPL", "MSFT", "NVDA"],
    ["fy_end", "be_filed", "me_date", "price", "book_equity", "market_cap", "bm", "excluded_reason"]]

**Market cap sanity check** (end of 2022, roughly): NVDA ≈ $0.36T, AAPL ≈ $2.1T, MSFT ≈ $1.8T, GOOGL ≈ $1.1T, META ≈ $0.3T. If GOOGL or META is far too low, yfinance is counting only one share class.

In [ ]:
(v23.set_index("ticker").loc[["NVDA", "AAPL", "MSFT", "GOOGL", "META"], "market_cap"] / 1e12).round(2)

### 2.7 All ranking dates → save the value signal table

In [ ]:
value_signals = pd.concat(
    [value_signal(t, tickers, be_hist, shares_hist, splits_hist, prices,
                  shares_split_adjusted=SHARES_SPLIT_ADJUSTED) for t in ranking_dates],
    ignore_index=True)
value_signals.to_csv(f"{SIGNALS}/value_signals.csv", index=False)
value_signals.head()

### 2.8 Sanity checks
**Exclusions per date.** Expect 3 banks + 2 foreign every year; MCD negative every year; STX negative from 2024; ORCL and HD negative in 2023; RKLB missing in 2021.

In [ ]:
value_signals.groupby(["rebalance_date", "excluded_reason"], dropna=False).size().unstack(fill_value=0)

In [ ]:
value_signals[value_signals["excluded_reason"].notna() & ~value_signals["ticker"].isin(["JPM","BAC","WFC","TSM","ASML"])] \
    [["rebalance_date", "ticker", "excluded_reason"]].sort_values(["rebalance_date", "ticker"])

**Direction check.** Cheap (high B/M) should look like energy, pharma, older industrials; expensive should look like high-growth tech.

In [ ]:
for t, g in value_signals.dropna(subset=["rank_pct"]).groupby("rebalance_date"):
    g = g.sort_values("rank_pct")
    print(t.date(), "| expensive:", list(g["ticker"].head(5)), "| cheap:", list(g["ticker"].tail(5)))

**Winsorizing.** Compare raw vs winsorized B/M per date.

In [ ]:
value_signals.groupby("rebalance_date")[["bm", "bm_w"]].describe().round(3)

### 2.9 Value signal: notes

- **Score:** B/M = book equity ÷ market cap. Book equity is from each firm's fiscal year ending in calendar year t−1 (52/53-week years ending in the first two weeks of January, e.g. JNJ's FY2022 ending 2023-01-01, count as the previous year), as originally reported in its 10-K (SEC EDGAR), and used only if filed on or before t. Market cap = shares outstanding × actual traded price on the last trading day of December t−1.
- **Price and shares on the same basis:** yfinance share counts are actual historical counts (confirmed: every real split in the data shows the count jumping by the split ratio). yfinance `close` is split-adjusted backwards, so it is converted back to the actual traded price using split history. `adj_close` is not used because it also removes dividends.
- **Ranking dates:** last trading day of June, 2021–2026. `rank_pct` runs from near 0 (most expensive) to 1 (cheapest) within each date.
- **Exclusions (flagged, not ranked):** banks (B/M not comparable), TSM/ASML (non-USD reporting), negative book equity (MCD every year; HD 2021 and 2023; ORCL 2023; STX 2024–2026), no FY t−1 report, missing price/shares (RKLB before listing; META 2021–2022, since yfinance share history only starts at the FB→META ticker change in June 2022), and share counts older than 18 months. *[Update with the final counts table above.]*
- **Stale share counts:** yfinance share history has gaps (e.g. NVDA has no observations from July 2021 to April 2023). Share counts change slowly, so counts up to 18 months old are used, with any split in between applied; older counts are flagged.
- **Winsorizing:** B/M clipped at the 5th/95th percentile within each date; with ~35 stocks, 1%/99% would clip nothing. For a single metric this does not change rank order; it matters if B/M is later combined with other metrics.
- **Limitations:** yfinance share counts are noisy and gappy; book equity uses the parent-company label where available and total equity (incl. minority interest) otherwise; ~35 stocks gives ~7 per quintile; the universe was chosen with hindsight.
- **Proxy:** none used; all scores use reported fundamentals. *[Change if any stock needed a proxy.]*